In [10]:
import pandas as pd

In [11]:
# add a new column called "Toxin" and set it to True
tox = pd.read_csv("../data/raw/0800.tsv", sep="\t")
tox["Toxin"] = True

# add a new column called "Toxin" and set it to False
nontox = pd.read_csv("../data/raw/nontox.tsv", sep="\t")
nontox["Toxin"] = False

# merge into a single DataFrame
tsv = pd.concat([tox, nontox], ignore_index=True)
tsv.dropna(subset=["Protein families"])
tsv.head()

,Entry,Protein families,Organism (ID),Sequence,Signal peptide,Toxin
0,A0A088MIT0,"Frog skin active peptide (FSAP) family, Bradyk...",248869,MAFLKKSLFLVLFLGVVSLSFCEEEKREEHEEEKRDEEDAESLGKR...,"SIGNAL 1..22; /evidence=""ECO:0000255""",True
1,A0A0B4U9L8,"Venom metalloproteinase (M12B) family, P-III s...",8705,MLQVLLVTICLAVFPYQGSSIILESGNVNDYEVVYPQKLTALLKGA...,"SIGNAL 1..20; /evidence=""ECO:0000255""",True
2,A0A0B5A8P4,Insulin family,6491,MTTSFYFLLVALGLLLYVCQSSFGNQHTRNSDTPKHRCGSELADQY...,"SIGNAL 1..21; /evidence=""ECO:0000255""",True
3,A0A0B5AC95,Insulin family,6491,MTTSSYFLLMALGLLLYVCQSSFGNQHTRTFDTPKHRCGSEITNSY...,"SIGNAL 1..24; /evidence=""ECO:0000255""",True
4,A0A0D4WV12,"Arthropod phospholipase D family, Class II sub...",571544,GDSRRPIWNIAHMVNDLDLVDEYLDDGANSLELDVEFSKSGTALRT...,NaN,True


In [12]:
# apply the same family/superfamily normalization as in preprocessing.load_and_prepare_raw
# first, normalize the raw "Protein families" column
tsv["Protein families"] = tsv["Protein families"].str.split(";").str[0]
tsv["Protein families"] = tsv["Protein families"].str.split(",").str[0]

repl = {
    "I1 superfamily": "Conotoxin I1 superfamily",
    "O1 superfamily": "Conotoxin O1 superfamily",
    "O2 superfamily": "Conotoxin O2 superfamily",
    "E superfamily": "Conotoxin E superfamily",
    "F superfamily": "Conotoxin F superfamily",
}
tsv["Protein families"] = tsv["Protein families"].replace(repl)

mapping = {
    r"Conotoxin.*": "Conotoxin family",
    r"Neurotoxin.*": "Neurotoxin family",
    r"Scoloptoxin.*|Scolopendra.*": "Scoloptoxin family",
    r"Caterpillar.*": "Caterpillar family",
    r"Teretoxin.*": "Teretoxin family",
    r"Limacoditoxin.*": "Limacoditoxin family",
    r"Scutigerotoxin.*": "Scutigerotoxin family",
    r"Cationic peptide.*": "Cationic peptide family",
    r"Formicidae venom.*": "Formicidae venom family",
    r"Sea anemone.*potassium channel toxin family.*": "Sea anemone potassium channel toxin family",
    r"Bradykinin-potentiating peptide family|Natriuretic peptide family|Natriuretic": "Natriuretic, Bradykinin potentiating peptide family",
    r".*phospholipase.*|.*Phospholipase.*": "Phospholipase family",
    r"Peptidase.*": "Peptidase family",
    r"FARP.*": "FARP family"
}

for pattern, replacement in mapping.items():
    tsv["Protein families"] = tsv["Protein families"].str.replace(
        pattern, replacement, regex=True
    )

# rename all that have 10 or less than 10 samples in "Protein families" into "Other"
vc = tsv["Protein families"].value_counts()

tsv["Protein families"] = tsv["Protein families"].where(
    tsv["Protein families"].map(vc) > 20,
    other="Other"
)

print(len(tsv["Protein families"].unique()))

# save all unique values in the Protein family column in a tsv file for Toxin column values that are True and False each
tsv[tsv["Toxin"] == True]["Protein families"].value_counts().to_csv("./Toxin_Pfams.csv")
tsv[tsv["Toxin"] == False]["Protein families"].value_counts().to_csv("./Non_Toxin_Pfams.csv")
tsv["Protein families"].value_counts().to_csv("./Pfams.csv")

684


In [13]:
import difflib

# unique families for toxin / non-toxin (after your normalization)
tox_fams = sorted(tsv.loc[tsv["Toxin"] == True, "Protein families"].dropna().unique())
nontox_fams = sorted(tsv.loc[tsv["Toxin"] == False, "Protein families"].dropna().unique())

# precompute counts for each family in toxin / non-toxin parts
tox_counts = tsv.loc[tsv["Toxin"] == True, "Protein families"].value_counts()
nontox_counts = tsv.loc[tsv["Toxin"] == False, "Protein families"].value_counts()

def best_match(name, candidates):
    best, best_score = None, 0.0
    for cand in candidates:
        score = difflib.SequenceMatcher(None, name, cand).ratio()
        if score > best_score:
            best, best_score = cand, score
    return best, best_score

rows = []
threshold = 0.88  # tune this (0–1) for stricter/looser matching

for fam in nontox_fams:
    match, score = best_match(fam, tox_fams)
    if score >= threshold:
        rows.append(
            (
                fam,
                match,
                score,
                nontox_counts.get(fam, 0),
                tox_counts.get(match, 0),
            )
        )

similar_non_tox = pd.DataFrame(
    rows,
    columns=[
        "Non-toxin family",
        "Closest toxin family",
        "similarity",
        "Non-toxin count",
        "Toxin count",
    ],
).sort_values("similarity", ascending=False)


similar_non_tox

,Non-toxin family,Closest toxin family,similarity,Non-toxin count,Toxin count
0,AB hydrolase superfamily,AB hydrolase superfamily,1.000000,258,2
25,Multicopper oxidase family,Multicopper oxidase family,1.000000,19,4
27,NPY family,NPY family,1.000000,104,4
28,"Natriuretic, Bradykinin potentiating peptide f...","Natriuretic, Bradykinin potentiating peptide f...",1.000000,68,100
29,Neurotoxin family,Neurotoxin family,1.000000,9,962
30,Non-disulfide-bridged peptide (NDBP) superfamily,Non-disulfide-bridged peptide (NDBP) superfamily,1.000000,138,33
31,Nucleotide pyrophosphatase/phosphodiesterase f...,Nucleotide pyrophosphatase/phosphodiesterase f...,1.000000,32,5
32,Other,Other,1.000000,48126,741
33,PBP/GOBP family,PBP/GOBP family,1.000000,76,16
34,PDGF/VEGF growth factor family,PDGF/VEGF growth factor family,1.000000,51,18


In [14]:
tsv

,Entry,Protein families,Organism (ID),Sequence,Signal peptide,Toxin
0,A0A088MIT0,Frog skin active peptide (FSAP) family,248869,MAFLKKSLFLVLFLGVVSLSFCEEEKREEHEEEKRDEEDAESLGKR...,"SIGNAL 1..22; /evidence=""ECO:0000255""",True
1,A0A0B4U9L8,Venom metalloproteinase (M12B) family,8705,MLQVLLVTICLAVFPYQGSSIILESGNVNDYEVVYPQKLTALLKGA...,"SIGNAL 1..20; /evidence=""ECO:0000255""",True
2,A0A0B5A8P4,Insulin family,6491,MTTSFYFLLVALGLLLYVCQSSFGNQHTRNSDTPKHRCGSELADQY...,"SIGNAL 1..21; /evidence=""ECO:0000255""",True
3,A0A0B5AC95,Insulin family,6491,MTTSSYFLLMALGLLLYVCQSSFGNQHTRTFDTPKHRCGSEITNSY...,"SIGNAL 1..24; /evidence=""ECO:0000255""",True
4,A0A0D4WV12,Phospholipase family,571544,GDSRRPIWNIAHMVNDLDLVDEYLDDGANSLELDVEFSKSGTALRT...,NaN,True
...,...,...,...,...,...,...
105768,Q9Y0Y7,Other,7227,MERRYLKNPFPDFAGGENTPFASDEEHIKNLICTYVDAILEHCHPN...,NaN,False
105769,Q9Y3F1,Other,9606,MSLLWTPQILTISFVSYILSLFPSPFPSCYTSCWFETSITTEKELN...,NaN,False
105770,Q9Y4M8,Other,9606,MATFHRAHATSSVKPRARRHQEPNSGDWPGSYRAGTRCSAIGFRLL...,NaN,False
105771,Q9Y6C7,Other,9606,MAHHSLNTFYIWHNNVLHTHLVFFLPHLLNQPFSRGSFLIWLLLCW...,NaN,False
